# Praktikum 1 – Regex‑basierter Tokenizer für deutsche Texte

Dieses Notebook demonstriert einen **einfachen, aber erweiterten Tokenizer**, der deutsche Texte mittels regulärer Ausdrücke zerlegt und anschließend –– durch Python‑Logik –– bestimmte Token zusammenführt.

**Haupterweiterungen gegenüber der ersten Version**  
* Titel wie `Dr.` oder `Mr.` bleiben mit dem folgenden Namen zusammen (`Dr. Brown`).  
* Mehrteilige Eigennamen werden erkannt (`New York`).  
* Datumsangaben (Bsp. `19.2.2000`) werden als *ein* Token behandelt.  
* Der Code bleibt reines Python + `re` (kein externes NLP‑Framework).

## Ablauf

1. **Text wählen** (hier: kurzer Wikipedia‑Artikelausschnitt).  
2. **Regex‑Pattern definieren** und auf den Text anwenden (`basic_tokenizer`).  
3. **Post‑Processing**: zusammenfassen von Token‑Folgen, z. B. Titel + Name, Datum, Eigennamen (`final_merge`).  
4. **Ergebnis anzeigen**: Vergleich *roh* vs. *gemergte* Token.

In [ ]:
import re

text = (
    "Bitcoin ist die erste und weltweit am Markt stärkste Kryptowährung. "
    "Sie nutzt ein dezentrales Buchungssystem auf Basis einer Blockchain.[5][6] "
    "Zahlungen werden kryptographisch legitimiert (digitale Signatur) und "
    "über ein Rechnernetz gleichberechtigter Computer (Peer-to-Peer) abgewickelt. "
    "Dr. Brown mag New York. Mr. Smith liebt (Digitale Kunst). "
    "Er wurde am 19.2.2000 geboren."
)

## Regex‑Pattern

```text
\[[^\]]*\]   – Eckige Klammern samt Inhalt (z. B. [5])  
\([^)]*\)     – Runde Klammern samt Inhalt (z. B. (digitale Signatur))  
[A-Za-zÄÖÜäöüß0-9\.]+ – Wörter/Zahlen (inkl. Punkt, damit `Dr.` erfasst wird)  
[^\w\s]       – Einzelne Satz‑/Sonderzeichen
```

In [ ]:
def basic_tokenizer(text: str):
    """Zerlegt den Text mithilfe eines Regex in Roh‑Token."""
    pattern = r'\[[^\]]*\]|\([^)]*\)|[A-Za-zÄÖÜäöüß0-9\.]+|[^\w\s]'
    return re.findall(pattern, text)

## Post‑Processing – `final_merge`

* **Titel + Name**: `Dr.` → `Dr. Brown`  
* **Datum**: `19 . 2 . 2000` oder `19.2.2000` → `19.2.2000`  
* **Zwei konsekutive Wörter mit Großbuchstaben** → zusammensetzen (`New York`).

In [ ]:
def final_merge(tokens):
    merged = []
    i = 0
    TITLES = {"Dr.", "Mr.", "Mrs.", "Ms.", "Prof."}

    while i < len(tokens):
        tok = tokens[i]

        # Titel + Name
        if tok in TITLES and i + 1 < len(tokens) and tokens[i + 1].istitle():
            merged.append(tok + " " + tokens[i + 1])
            i += 2
            continue

        # Datum 19.2.2000  /  19 . 2 . 2000
        if re.fullmatch(r"\d{1,2}", tok):
            # Variante 1: 19.2.2000 (ein Token  ,  d.h. nach Zahl direkt '.')
            if (i + 2 < len(tokens)
                and tokens[i + 1] == '.'
                and re.fullmatch(r"\d{1,2}", tokens[i + 2])):
                # Prüfen auf vierstellige Jahreszahl dahinter
                if (i + 4 < len(tokens)
                    and tokens[i + 3] == '.'
                    and re.fullmatch(r"\d{4}", tokens[i + 4])):
                    merged.append(f"{tok}.{tokens[i + 2]}.{tokens[i + 4]}")
                    i += 5
                    continue
                elif re.fullmatch(r"\d{4}", tokens[i + 2]):
                    merged.append(f"{tok}.{tokens[i + 2]}")
                    i += 3
                    continue

        # Zwei aufeinanderfolgende Groß‑Tokens → New York
        if tok.istitle() and i + 1 < len(tokens) and tokens[i + 1].istitle():
            merged.append(tok + " " + tokens[i + 1])
            i += 2
            continue

        # Standardfall
        merged.append(tok)
        i += 1

    return merged

## Anwendung des Tokenizers auf den Beispieltext

In [ ]:
raw_tokens = basic_tokenizer(text)
final_tokens = final_merge(raw_tokens)
print("Tokens nach basic_tokenizer (erste 40):\n", raw_tokens[:40], "...\n")
print("\nFinale Tokens nach final_merge:\n", final_tokens)

### Fazit

Der reine Regex‑Ansatz liefert **gute, aber nicht perfekte Ergebnisse**.  
Für produktive Anwendungen wäre ein vollwertiger NLP‑Tokenizer (z. B. spaCy de) robuster – jedoch zeigt dieses Notebook, wie man **rein mit Python‑Bordmitteln** viele Problemfälle schon abfangen kann.